# Lab 07 - Đánh giá mô hình

Sinh viên: **Lê Quyết Tiến** - MSSV: **2474802010386**.

Notebook chạy từ thư mục gốc `07. Accuracy_Metrics`. Trên Colab, hãy mount Google Drive, `%cd` đến thư mục này rồi chọn **Runtime > Run all**. Giữ nguyên hai tệp CSV trong `data/`.


## Bước 1 - Khi accuracy đánh lừa


In [1]:
# -*- coding: utf-8 -*-
"""Bước 1: thấy accuracy đánh lừa trên dữ liệu lệch lớp."""

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
print("Kich thuoc bang:", df.shape)
print(df.head())
print()
print("So sinh vien moi lop:")
print(df["bo_hoc"].value_counts().to_string())
print(f"Ty le bo hoc: {df['bo_hoc'].mean():.3f}")
print()

X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print(f"Tap kiem tra: {len(y_te)} ban, {y_te.sum()} ban bo hoc")
print()

luoi = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
du_luoi = luoi.predict(X_te)
lr = make_pipeline(StandardScaler(), LogisticRegression())
du_lr = lr.fit(X_tr, y_tr).predict(X_te)

for ten, du in [("Mo hinh luoi", du_luoi), ("Hoi quy logistic", du_lr)]:
    acc = accuracy_score(y_te, du)
    tim_ra = ((du == 1) & (y_te == 1)).sum()
    print(f"{ten:17s} accuracy {acc:.4f}, tim ra {tim_ra} tren {y_te.sum()} ban bo hoc")


Kich thuoc bang: (1200, 5)
   diem_tb  so_buoi_vang  tin_chi_no  gio_lam_them  bo_hoc
0      8.0             3           0             0       0
1      8.9             5           1             7       0
2      3.3             3           1             4       0
3      6.4             3           0             7       0
4      7.9             3           5            23       0

So sinh vien moi lop:
bo_hoc
0    1081
1     119
Ty le bo hoc: 0.099

Tap kiem tra: 300 ban, 30 ban bo hoc

Mo hinh luoi      accuracy 0.9000, tim ra 0 tren 30 ban bo hoc
Hoi quy logistic  accuracy 0.9467, tim ra 17 tren 30 ban bo hoc


## Bước 2 - Ma trận nhầm lẫn


In [2]:
# -*- coding: utf-8 -*-
"""Bước 2: tự đếm bốn ô của ma trận nhầm lẫn, rồi so với thư viện."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
lr = make_pipeline(StandardScaler(), LogisticRegression())
du = lr.fit(X_tr, y_tr).predict(X_te)
that = y_te.to_numpy()

tp = np.sum((that == 1) & (du == 1))
fp = np.sum((that == 0) & (du == 1))
fn = np.sum((that == 1) & (du == 0))
tn = np.sum((that == 0) & (du == 0))
print("Tu dem:")
print(f" TP = {tp} (doan bo hoc, dung la bo hoc)")
print(f" FP = {fp} (bao dong nham)")
print(f" FN = {fn} (bo sot)")
print(f" TN = {tn} (doan khong bo, dung la khong bo)")
print(f" Tong = {tp + fp + fn + tn}")
print()

m = confusion_matrix(that, du)
print("confusion_matrix cua scikit-learn:")
print(m)
print("Thu tu khi trai ra: TN, FP, FN, TP =", m.ravel())
print()
acc = (tp + tn) / (tp + fp + fn + tn)
print(f"Accuracy tinh tu bon o: ({tp} + {tn}) / {len(that)} = {acc:.4f}")
print(f"So ban bi bao co nguy co: TP + FP = {tp + fp}")


Tu dem:
 TP = 17 (doan bo hoc, dung la bo hoc)
 FP = 3 (bao dong nham)
 FN = 13 (bo sot)
 TN = 267 (doan khong bo, dung la khong bo)
 Tong = 300

confusion_matrix cua scikit-learn:
[[267   3]
 [ 13  17]]
Thu tu khi trai ra: TN, FP, FN, TP = [267   3  13  17]

Accuracy tinh tu bon o: (17 + 267) / 300 = 0.9467
So ban bi bao co nguy co: TP + FP = 20


## Bước 3 - Precision, recall và F1


In [3]:
# -*- coding: utf-8 -*-
"""Bước 3: precision, recall, F1 tính tay và tính bằng thư viện."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, precision_score, recall_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
lr = make_pipeline(StandardScaler(), LogisticRegression())
du = lr.fit(X_tr, y_tr).predict(X_te)
tn, fp, fn, tp = confusion_matrix(y_te, du).ravel()

p = tp / (tp + fp)
r = tp / (tp + fn)
f1 = 2 * p * r / (p + r)
print("Tinh tay tu bon o:")
print(f" Precision = {tp} / ({tp} + {fp}) = {p:.4f}")
print(f" Recall = {tp} / ({tp} + {fn}) = {r:.4f}")
print(f" F1 = 2 x {p:.4f} x {r:.4f} / ({p:.4f} + {r:.4f}) = {f1:.4f}")
print()
print("Tinh bang scikit-learn:")
print(f" Precision = {precision_score(y_te, du):.4f}")
print(f" Recall = {recall_score(y_te, du):.4f}")
print(f" F1 = {f1_score(y_te, du):.4f}")
print()
print("Bao cao day du:")
print(classification_report(y_te, du, digits=4, target_names=["khong bo", "bo hoc"]))


Tinh tay tu bon o:
 Precision = 17 / (17 + 3) = 0.8500
 Recall = 17 / (17 + 13) = 0.5667
 F1 = 2 x 0.8500 x 0.5667 / (0.8500 + 0.5667) = 0.6800

Tinh bang scikit-learn:
 Precision = 0.8500
 Recall = 0.5667
 F1 = 0.6800

Bao cao day du:
              precision    recall  f1-score   support

    khong bo     0.9536    0.9889    0.9709       270
      bo hoc     0.8500    0.5667    0.6800        30

    accuracy                         0.9467       300
   macro avg     0.9018    0.7778    0.8255       300
weighted avg     0.9432    0.9467    0.9418       300



## Bước 4 - Đổi và chọn ngưỡng


In [4]:
# -*- coding: utf-8 -*-
"""Bước 4: đổi ngưỡng, và chọn ngưỡng đúng cách trên tập học."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
lr = make_pipeline(StandardScaler(), LogisticRegression())
lr.fit(X_tr, y_tr)
xs = lr.predict_proba(X_te)[:, 1]

print("Tap kiem tra, doi nguong")
print("nguong so bao  TP  FP  FN  precision recall F1")
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    du = (xs >= t).astype(int)
    tp = int(((du == 1) & (y_te == 1)).sum())
    fp = int(((du == 1) & (y_te == 0)).sum())
    fn = int(((du == 0) & (y_te == 1)).sum())
    print(
        f" {t:.1f}    {du.sum():4d} {tp:3d} {fp:3d} {fn:3d}"
        f"   {precision_score(y_te, du):.4f}   {recall_score(y_te, du):.4f} {f1_score(y_te, du):.4f}"
    )
print()

xs_hoc = cross_val_predict(lr, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]
cac_nguong = np.round(np.arange(0.05, 0.96, 0.05), 2)
f1_hoc = [f1_score(y_tr, (xs_hoc >= t).astype(int)) for t in cac_nguong]
t_tot = cac_nguong[int(np.argmax(f1_hoc))]
print(f"Nguong cho F1 cao nhat tren tap hoc: {t_tot} (F1 kiem dinh cheo {max(f1_hoc):.4f})")
du = (xs >= t_tot).astype(int)
print(
    f"Tap kiem tra voi nguong {t_tot}: precision {precision_score(y_te, du):.4f},"
    f" recall {recall_score(y_te, du):.4f}, F1 {f1_score(y_te, du):.4f}"
)


Tap kiem tra, doi nguong
nguong so bao  TP  FP  FN  precision recall F1
 0.5      20  17   3  13   0.8500   0.5667 0.6800
 0.4      21  18   3  12   0.8571   0.6000 0.7059
 0.3      29  21   8   9   0.7241   0.7000 0.7119
 0.2      42  24  18   6   0.5714   0.8000 0.6667
 0.1      61  26  35   4   0.4262   0.8667 0.5714

Nguong cho F1 cao nhat tren tap hoc: 0.25 (F1 kiem dinh cheo 0.6473)
Tap kiem tra voi nguong 0.25: precision 0.6571, recall 0.7667, F1 0.7077


## Bước 5 - Đường ROC và AUC


In [5]:
# -*- coding: utf-8 -*-
"""Bước 5: đường ROC, AUC, và so các mô hình đã học trong môn."""

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
lr = make_pipeline(StandardScaler(), LogisticRegression())
xs = lr.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
fpr, tpr, nguong = roc_curve(y_te, xs)
print("Duong ROC co", len(fpr), "diem. Vai diem dau:")
print(" bao dong nham  recall")
for a, b in list(zip(fpr, tpr))[:6]:
    print(f" {a:.4f}          {b:.4f}")
print(f"AUC cua logistic = {roc_auc_score(y_te, xs):.4f}")
print()

mo_hinh = {
    "Logistic": lr,
    "Naive Bayes": GaussianNB(),
    "KNN, k = 15": make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
    "Cay sau 4": DecisionTreeClassifier(max_depth=4, random_state=0),
    "Rung 200 cay": RandomForestClassifier(200, random_state=0),
    "SVM RBF": make_pipeline(StandardScaler(), SVC(probability=True, random_state=0)),
}
print("Mo hinh          AUC    recall   F1 (nguong 0.5)")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    p = mh.predict_proba(X_te)[:, 1]
    du = (p >= 0.5).astype(int)
    print(f" {ten:14s} {roc_auc_score(y_te, p):.4f} {recall_score(y_te, du):.4f} {f1_score(y_te, du):.4f}")


Duong ROC co 28 diem. Vai diem dau:
 bao dong nham  recall
 0.0000          0.0000
 0.0000          0.0333
 0.0000          0.4000
 0.0074          0.4000
 0.0074          0.5333
 0.0111          0.5333
AUC cua logistic = 0.9449

Mo hinh          AUC    recall   F1 (nguong 0.5)
 Logistic       0.9449 0.5667 0.6800
 Naive Bayes    0.9336 0.3667 0.5116
 KNN, k = 15    0.9274 0.2333 0.3684
 Cay sau 4      0.8803 0.4667 0.5957
 Rung 200 cay   0.9249 0.5667 0.6538
 SVM RBF        0.8996 0.5000 0.6250


## Bước 6 - Kiểm định chéo và cân bằng lớp


In [6]:
# -*- coding: utf-8 -*-
"""Bước 6: kiểm định chéo với nhiều thước đo, và cân bằng lớp khi học."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
print("Khong stratify, so ban bo hoc trong tap kiem tra:")
dem = []
for rs in range(10):
    _, _, _, y_te_thu = train_test_split(X, y, test_size=0.25, random_state=rs)
    dem.append(int(y_te_thu.sum()))
print(" ", dem)
print()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
lr = make_pipeline(StandardScaler(), LogisticRegression())
print("Kiem dinh cheo 5 phan tren tap hoc:")
for tm in ["accuracy", "precision", "recall", "f1"]:
    d = cross_val_score(lr, X_tr, y_tr, cv=5, scoring=tm)
    print(f" {tm:10s} {d.mean():.4f}")
print()

cb = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced"))
print("Tap kiem tra  precision recall F1")
for ten, mh in [("Binh thuong", lr), ("Can bang lop", cb)]:
    du = mh.fit(X_tr, y_tr).predict(X_te)
    print(f" {ten:12s} {precision_score(y_te, du):.4f}    {recall_score(y_te, du):.4f} {f1_score(y_te, du):.4f}")


Khong stratify, so ban bo hoc trong tap kiem tra:
  [22, 26, 35, 29, 32, 27, 28, 20, 27, 23]

Kiem dinh cheo 5 phan tren tap hoc:
 accuracy   0.9289
 precision  0.7044
 recall     0.4810
 f1         0.5652

Tap kiem tra  precision recall F1
 Binh thuong  0.8500    0.5667 0.6800
 Can bang lop 0.4483    0.8667 0.5909
